# Person 2 — Support Vector Machine & Data Preprocessing Pipeline

Pipeline Responsibility: Data Preprocessing, Audit & Split\nModel Assignment: Support Vector Machine (SVM)

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.svm import SVC

print("--- Person 2: Support Vector Machine (SVM) Model Training & Data Preprocessing ---")

# Compare Linear vs RBF kernels
kernels = ['linear', 'rbf']
best_svm = None
best_f1 = 0.0

for k in kernels:
    svm_pipe = make_pipeline(StandardScaler(), SVC(C=1.0, kernel=k, class_weight='balanced', random_state=SEED))
    svm_pipe.fit(X_tr, y_tr)
    val_preds = svm_pipe.predict(X_te)
    score = f1_score(y_te, val_preds, average='macro')
    print(f"Kernel '{k}' Macro F1: {score:.4f}")
    if score > best_f1:
        best_f1 = score
        best_svm = svm_pipe

start_time = time.perf_counter()
best_svm.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = best_svm.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

svc_clf = best_svm.named_steps['svc']
n_support = svc_clf.n_support_.tolist()

print(f"Best SVM Model Kernel: {svc_clf.kernel}")
print(f"Number of Support Vectors per class: {n_support}")
print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": f"SVM ({svc_clf.kernel} kernel)",
    "pipeline_stage": "Data Preprocessing & Split",
    "kernel": svc_clf.kernel,
    "support_vectors_per_class": n_support,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "svm_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(best_svm, OUTPUT_DIR / "svm_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)


--- Person 2: Support Vector Machine (SVM) Model Training & Data Preprocessing ---


Kernel 'linear' Macro F1: 0.9167


Kernel 'rbf' Macro F1: 0.9500


Best SVM Model Kernel: rbf
Number of Support Vectors per class: [290, 280]
Accuracy: 0.9500 | Macro F1: 0.9500
Confusion Matrix:
 [[88  4]
 [ 5 83]]
Saved outputs to: D:\SLIIT\projectr\Dataset_Train\parts\svm\outputs
